<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/IPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import json
import time
import io
import requests
import pandas as pd
import yfinance as yf


### Loading and Merging 'IPO Deal Size' Data

Now, let's load the provided `IPO_Deal_Size.csv` file and merge its information (Deal Size) into our `consolidated_df`. We will assume the 'Symbol' column is present in both DataFrames for the merge operation.

In [39]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
IPO_2024_FileId = '1CiGIOc7Z3Td4xTCICy7H3cF0QBIeUFiu'
IPO_2024_File = f'https://drive.google.com/uc?export=download&id={IPO_2024_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2024_df = pd.read_csv(IPO_2024_File, sep=',', quotechar='"', skipinitialspace=True)

IPO_2025_FileId = '1uSI7gY4uKACxZqipBj3SefHSouAaNfrr'
IPO_2025_File = f'https://drive.google.com/uc?export=download&id={IPO_2025_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2025_df = pd.read_csv(IPO_2025_File, sep=',', quotechar='"', skipinitialspace=True)

IPO_2026_FileId = '1XjCKzN_pXzoawIGpFqQs2iAtdBOBbvJC'
IPO_2026_File = f'https://drive.google.com/uc?export=download&id={IPO_2026_FileId}'
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_2026_df = pd.read_csv(IPO_2026_File, sep=',', quotechar='"', skipinitialspace=True)

In [40]:
# Concatenate the three IPO DataFrames into a single DataFrame
consolidated_df_with_deal_size = pd.concat([ipo_2024_df, ipo_2025_df, ipo_2026_df], ignore_index=True)

print("--- Consolidated DataFrame with Deal Size (Head) ---")
display(consolidated_df_with_deal_size.head())

print("--- Consolidated DataFrame with Deal Size (Tail) ---")
display(consolidated_df_with_deal_size.tail())

print("\nMerge complete. The `consolidated_df_with_deal_size` DataFrame now includes IPO data from 2024, 2025, and 2026.")

--- Consolidated DataFrame with Deal Size (Head) ---


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$1.20,-71.00%,NASDAQ,7.00M,"1,750,000",$4.00
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",$0.435,-,NASDAQ,-,-,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$1.03,-73.50%,NASDAQ,9.20M,"2,300,000",$4.00
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.70,7.00%,NASDAQ,150.00M,"15,000,000",$10.00
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.67,6.70%,NASDAQ,100.00M,"10,000,000",$10.00


--- Consolidated DataFrame with Deal Size (Tail) ---


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
766,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.32,3.20%,NASDAQ,"25,000,000",250.00M,$10.00
767,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$10.80,43.20%,NYSEAMERICAN,"2,666,667",20.00M,$7.50
768,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$9.94,-0.60%,NASDAQ,"22,000,000",220.00M,$10.00
769,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$9.98,-0.20%,NASDAQ,"22,000,000",220.00M,$10.00
770,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$9.94,-0.40%,NYSE,"15,000,000",150.00M,$10.00



Merge complete. The `consolidated_df_with_deal_size` DataFrame now includes IPO data from 2024, 2025, and 2026.


### Analyze and Calculate Returns for Large Deal Sizes

First, I will filter the `consolidated_df_with_deal_size` DataFrame to include only those IPOs where the 'Deal Size' contains 'B' (indicating billions). Then, for each of these filtered IPOs, I will use the `yfinance` library to fetch the stock's historical data for the IPO date. Finally, I will calculate the return based on the 'Open' and 'Close' prices for that specific date.

In [47]:
import pandas as pd
import yfinance as yf
from datetime import timedelta, datetime # Import datetime

# Filter for rows where 'Deal Size' contains 'B'
# Ensure 'IPO Date' column is converted to datetime if not already
temp_df = consolidated_df_with_deal_size[consolidated_df_with_deal_size['Deal Size'].str.contains('B', na=False)].copy()
temp_df['IPO Date'] = pd.to_datetime(temp_df['IPO Date'])

# Filter out IPOs that are in the future
today = datetime.now().date() # Get only the date part of today
filtered_df = temp_df[temp_df['IPO Date'].dt.date <= today].copy() # Compare only date parts

# Prepare empty lists for 1, 3, 5, 10, 20, 30 day returns
returns_1d = []
returns_3d = []
returns_5d = []
returns_10d = []
returns_20d = []
returns_30d = []

print(f"Found {len(filtered_df)} *past* IPOs with 'B' in 'Deal Size'. Attempting to fetch historical stock data for post-IPO returns...")

for index, row in filtered_df.iterrows():
    symbol = row['Symbol']
    ipo_date = row['IPO Date']

    try:
        # Fetch data for a period long enough to cover 30 trading days after IPO
        # 60 calendar days should be sufficient to capture 30 trading days
        end_fetch_date = ipo_date + timedelta(days=60)

        # Download daily historical data
        ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
                                  end=end_fetch_date.strftime('%Y-%m-%d'),
                                  interval="1d", progress=False)

        # Check if ticker_data is not empty and contains 'Adj Close'
        if not ticker_data.empty and 'Adj Close' in ticker_data.columns:
            # The first entry in ticker_data is the first available trading day's data
            # This implicitly handles the 'first trading day after IPO' if IPO day has no data.
            first_trading_day_close = ticker_data['Adj Close'].iloc[0]
            first_trading_date = ticker_data.index[0]

            # Calculate returns for specific trading days
            trading_days_to_check = [
                ('Return_1_Trading_Day', 1),
                ('Return_3_Trading_Days', 3),
                ('Return_5_Trading_Days', 5),
                ('Return_10_Trading_Days', 10),
                ('Return_20_Trading_Days', 20),
                ('Return_30_Trading_Days', 30)
            ]

            current_returns = {label: None for label, _ in trading_days_to_check}

            for label, days_offset in trading_days_to_check:
                # Ensure there are enough trading days to get the offset price
                if len(ticker_data) > days_offset:
                    price_at_offset = ticker_data['Adj Close'].iloc[days_offset]
                    if first_trading_day_close != 0: # Avoid division by zero
                        ret = (price_at_offset - first_trading_day_close) / first_trading_day_close
                        current_returns[label] = ret
                # If not enough trading days, it remains None as initialized

            returns_1d.append(current_returns['Return_1_Trading_Day'])
            returns_3d.append(current_returns['Return_3_Trading_Days'])
            returns_5d.append(current_returns['Return_5_Trading_Days'])
            returns_10d.append(current_returns['Return_10_Trading_Days'])
            returns_20d.append(current_returns['Return_20_Trading_Days'])
            returns_30d.append(current_returns['Return_30_Trading_Days'])

            print(f"    ✅ Fetched {symbol} (IPO Date: {ipo_date.strftime('%Y-%m-%d')}). First trading day: {first_trading_date.strftime('%Y-%m-%d')}.")

        else:
            print(f"    ❌ No valid historical stock data (or 'Adj Close' column missing) found for {symbol} starting from {ipo_date.strftime('%Y-%m-%d')} from Yahoo Finance.")
            # Append None for all return types if data is invalid or missing
            returns_1d.append(None)
            returns_3d.append(None)
            returns_5d.append(None)
            returns_10d.append(None)
            returns_20d.append(None)
            returns_30d.append(None)

    except Exception as e:
        print(f"    ❌ Error fetching data for {symbol} (IPO Date: {ipo_date.strftime('%Y-%m-%d')}): {e}")
        # Append None for all return types in case of an exception
        returns_1d.append(None)
        returns_3d.append(None)
        returns_5d.append(None)
        returns_10d.append(None)
        returns_20d.append(None)
        returns_30d.append(None)

# Add the calculated returns to the filtered DataFrame
filtered_df['Return_1_Trading_Day'] = returns_1d
filtered_df['Return_3_Trading_Days'] = returns_3d
filtered_df['Return_5_Trading_Days'] = returns_5d
filtered_df['Return_10_Trading_Days'] = returns_10d
filtered_df['Return_20_Trading_Days'] = returns_20d
filtered_df['Return_30_Trading_Days'] = returns_30d

# Remove old IPO_Day columns if they exist as they are no longer needed
if 'IPO_Day_Open' in filtered_df.columns:
    filtered_df = filtered_df.drop(columns=['IPO_Day_Open', 'IPO_Day_Close', 'IPO_Day_Return'])

Found 13 *past* IPOs with 'B' in 'Deal Size'. Attempting to fetch historical stock data for post-IPO returns...
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for SARO starting from 2024-10-02 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for LINE starting from 2024-07-25 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for VIK starting from 2024-05-01 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for AS starting from 2024-02-01 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for MDLN starting from 2025-12-17 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for BETA starting from 2025-11-04 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for KLAR starting from 2025-09-10 from Yah

/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF

    ❌ No valid historical stock data (or 'Adj Close' column missing) found for NIQ starting from 2025-07-23 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for CRCL starting from 2025-06-05 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for CRWV starting from 2025-03-28 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for SAIL starting from 2025-02-13 from Yahoo Finance.
    ❌ No valid historical stock data (or 'Adj Close' column missing) found for VG starting from 2025-01-24 from Yahoo Finance.


/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),
/tmp/ipykernel_3041/2503339970.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker_data = yf.download(symbol, start=ipo_date.strftime('%Y-%m-%d'),


In [42]:
print("--- Filtered DataFrame with IPO Day Returns ---")
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'Deal Size', 'IPO_Day_Open', 'IPO_Day_Close', 'IPO_Day_Return']].head())
display(filtered_df[['Symbol', 'Company Name', 'IPO Date', 'Deal Size', 'IPO_Day_Open', 'IPO_Day_Close', 'IPO_Day_Return']].tail())

--- Filtered DataFrame with IPO Day Returns ---


,Symbol,Company Name,IPO Date,Deal Size,IPO_Day_Open,IPO_Day_Close,IPO_Day_Return
69,SARO,"StandardAero, Inc.","Oct 2, 2024",1.44B,None,None,None
115,LINE,"Lineage, Inc.","Jul 25, 2024",4.44B,None,None,None
160,VIK,Viking Holdings Ltd,"May 1, 2024",1.27B,None,None,None
209,AS,"Amer Sports, Inc.","Feb 1, 2024",1.37B,None,None,None
234,MDLN,Medline Inc.,"Dec 17, 2025",6.26B,None,None,None


,Symbol,Company Name,IPO Date,Deal Size,IPO_Day_Open,IPO_Day_Close,IPO_Day_Return
380,NIQ,NIQ Global Intelligence plc,"Jul 23, 2025",1.05B,None,None,None
431,CRCL,"Circle Internet Group, Inc.","Jun 5, 2025",1.05B,None,None,None
499,CRWV,"CoreWeave, Inc.","Mar 28, 2025",1.50B,None,None,None
532,SAIL,"SailPoint, Inc.","Feb 13, 2025",1.38B,None,None,None
553,VG,"Venture Global, Inc.","Jan 24, 2025",1.75B,None,None,None
